# p01_faq_search

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/javanmardi298/rag_ollama/blob/main/lessons/p01_faq_search/p01_faq_search.ipynb)

**روال هر گام:** ۱) متن را بخوان ۲) کد را ببین و کامل کن (✍️) ۳) سلول را اجرا کن (Shift+Enter) ۴) «بررسی خودکار» (✅) را اجرا کن ۵) امتحان کوتاه (📝). در پایان «گزارش نهایی درس» نتیجه‌ی همه‌ی گام‌ها را نشان می‌دهد.

همه‌ی کد همین‌جا نوشته و اجرا می‌شود؛ به فایل جداگانه‌ای نیاز نیست. سلول‌ها را **به ترتیب** اجرا کن.

In [ ]:
# 1) آماده‌سازی (Colab: پروژه را clone و نصب می‌کند؛ محلی: اگر داخل پوشه‌ی پروژه‌ای چیزی تغییر نمی‌کند)
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', 'https://github.com/javanmardi298/rag_ollama.git', 'repo'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'repo/requirements.txt', '-e', 'repo'], check=True)
if os.path.exists('repo'):
    os.chdir('repo')
while not os.path.exists('lessons') and os.getcwd() != os.path.dirname(os.getcwd()):   # باز کردن از داخل lessons/...
    os.chdir(os.path.dirname(os.getcwd()))
print('پوشه‌ی کار:', os.getcwd())

### مدل زبانی درس‌ها
پیش‌فرض `offline` است (بدون نصب؛ شکل کار را نشان می‌دهد). برای جواب واقعی یکی از خط‌های زیر را از حالت توضیح (`#`) دربیاور. کلید API را **داخل notebook ننویس** (commit می‌شود)؛ از `.env` یا `getpass` استفاده کن.

In [ ]:
import os
os.environ['LEARN_PROVIDER'] = 'offline'      # 'offline' | 'ollama' | 'anthropic' | 'opencode' | 'openai_compatible'
# os.environ['LEARN_PROVIDER'] = 'ollama';     os.environ['LEARN_MODEL'] = 'qwen2.5:7b'
# os.environ['LEARN_PROVIDER'] = 'anthropic';  os.environ['LEARN_MODEL'] = 'claude-sonnet-5-5'; os.environ['LEARN_BUDGET_USD'] = '0.5'
#   import getpass; os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')   # اشتراک Claude.ai کلید API نیست
# os.environ['LEARN_PROVIDER'] = 'opencode';   os.environ['LEARN_MODEL'] = 'anthropic/claude-sonnet-5-5'
# os.environ['LEARN_PROVIDER'] = 'openai_compatible'; os.environ['LEARN_API_BASE'] = 'http://localhost:1234/v1'; os.environ['LEARN_MODEL'] = '...'
from learn import llm as learn_llm
reply = learn_llm.get().chat([{'role': 'user', 'content': 'Reply with the single word OK.'}])
print(os.environ['LEARN_PROVIDER'], '->', reply.content[:80], f'({reply.latency_s:.1f}s)')

---

# گام ۱: تکه‌تکه کردن سند (chunking)

## هدف این گام
در پایان این گام تابعی می‌نویسی که یک سند markdown را به تکه‌های کوچک و مستقل تقسیم می‌کند؛ هر تکه یک شناسه، عنوان بخش و متن دارد. این اولین قدم هر سیستم جست‌وجو و RAG است.

## پیش‌نیاز
فقط Python پایه (حلقه، رشته، لیست). نصب خاصی لازم نیست. اگر هنوز محیط را نساخته‌ای [راه‌اندازی](#) را بخوان.

## مفهوم: چرا chunk؟
وقتی کاربر می‌پرسد «چند روز وقت دارم کالا را برگردانم؟»، جواب در **یک جمله از یک صفحه** است، نه در کل سایت. اگر کل سند را به مدل زبانی بدهی:
- هزینه و زمان بالا می‌رود (هر توکن پول و ثانیه است)،
- مدل با اطلاعات نامرتبط گمراه می‌شود،
- نمی‌توانی بگویی جواب از کدام بخش آمده (ارجاع/citation ناممکن می‌شود).

پس سند را به تکه‌ها می‌شکنیم و فقط تکه‌های مرتبط را پیدا می‌کنیم. دو خطای رایج:
1. **تکه خیلی بزرگ:** جست‌وجو کم‌دقت می‌شود چون چند موضوع در یک تکه است.
2. **تکه خیلی کوچک:** زمینه (context) گم می‌شود. جمله‌ی «۳۰ روز» بدون اینکه بدانی درباره‌ی بازگشت کالاست بی‌معنی است.

راه حل این گام: **تقسیم بر اساس عنوان‌ها** (heading). هر بخش `##` یک تکه می‌شود و عنوان کامل (`Returns > Return window`) کنارش می‌ماند تا زمینه گم نشود. این روش برای سندهایی که ساختار دارند (FAQ، مستندات، سیاست‌ها) بهترین نقطه‌ی شروع است. روش‌های دیگر (اندازه‌ی ثابت با هم‌پوشانی، بازگشتی) در پروژه‌ی `labkit.chunking` هستند و در گام ۵ با آن‌ها مقایسه می‌کنیم.

## تمرین کد

تابع `chunk_markdown(text, doc_id)` را کامل کن:

| ورودی | معنی |
|---|---|
| `text` | متن markdown یک سند |
| `doc_id` | نام سند، مثلاً `returns` |

خروجی لیستی از دیکشنری‌هاست، با کلیدهای `id` (به شکل `returns#0`، `returns#1`، …)، `doc_id`، `heading` (مثل `Returns > Return window`) و `text` (متن بخش، بدون خط عنوان، با فاصله‌های اضافی تمیز شده).

قواعد:
- خط `# عنوان` عنوان سطح ۱ است؛ خط `## عنوان` سطح ۲.
- هر بار که عنوان جدیدی می‌رسد، تکه‌ی جاری بسته می‌شود.
- تکه‌ی بدون متن (مثلاً فقط یک عنوان `#` که بلافاصله `##` دارد) ساخته نمی‌شود.
- شماره‌ی `id` فقط تکه‌های ساخته‌شده را می‌شمارد.

### کد، خط‌به‌خط (الگوی حل)
1. دو متغیر `h1` و `h2` عنوان‌های جاری را نگه می‌دارند و `buf` خط‌های بخش جاری را جمع می‌کند.
2. تابع داخلی `flush()` خط‌های جمع‌شده را به یک متن تمیز تبدیل می‌کند (`" ".join(text.split())` فاصله‌ها و خط‌های جدید اضافی را یکی می‌کند) و اگر خالی نبود تکه می‌سازد.
3. حلقه روی خط‌ها: اگر خط با `## ` شروع شود `flush()` و بعد `h2` را به‌روز کن؛ اگر با `# ` شروع شود `flush()`، `h1` را به‌روز و `h2` را خالی کن؛ وگرنه خط را به `buf` اضافه کن.
4. بعد از حلقه یک `flush()` دیگر برای آخرین بخش.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 1: split a markdown document into chunks that keep their heading path."""
from __future__ import annotations


def chunk_markdown(text: str, doc_id: str) -> list[dict]:
    """Return [{"id": "returns#0", "doc_id": "returns", "heading": "Returns > Return window", "text": "..."}, ...]"""
    chunks: list[dict] = []
    h1, h2, buf = "", "", []

    def flush():
        # TODO 1: turn the lines in `buf` into one clean string (hint: " ".join(" ".join(buf).split()))
        #         if it is not empty, append a chunk dict to `chunks`, then clear `buf`.
        raise NotImplementedError("TODO 1: flush() builds a chunk from buf")

    for line in text.splitlines():
        # TODO 2: a line starting with "## " ends the current chunk and sets h2;
        #         a line starting with "# " ends the current chunk, sets h1 and resets h2;
        #         any other line belongs to the current chunk.
        raise NotImplementedError("TODO 2: walk through the lines")
    flush()
    return chunks

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
from learn.data import faq_docs
for name, text in faq_docs().items():
    for c in chunk_markdown(text, name):
        print(c["id"], "|", c["heading"], "|", c["text"][:50])

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p01_faq_search', 'step01_chunk_markdown', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
SAMPLE = "# Returns\n\n## Return window\nYou can return   items\nwithin 30 days.\n\n## Exceptions\nCustom items cannot be returned.\n"


def check_one_chunk_per_section(m):
    """one chunk per ## section"""
    cs = m.chunk_markdown(SAMPLE, "returns")
    assert len(cs) == 2, f"expected 2 chunks, got {len(cs)}"


def check_heading_path(m):
    """heading keeps 'Title > Section'"""
    cs = m.chunk_markdown(SAMPLE, "returns")
    assert cs[0]["heading"] == "Returns > Return window", f"got {cs[0]['heading']!r}"
    assert cs[1]["heading"] == "Returns > Exceptions", f"got {cs[1]['heading']!r}"


def check_text_is_clean(m):
    """text has no heading line and normalised spaces"""
    cs = m.chunk_markdown(SAMPLE, "returns")
    assert cs[0]["text"] == "You can return items within 30 days.", f"got {cs[0]['text']!r}"


def check_ids(m):
    """ids are doc_id#0, doc_id#1, ... and doc_id is stored"""
    cs = m.chunk_markdown(SAMPLE, "returns")
    assert [c["id"] for c in cs] == ["returns#0", "returns#1"], [c["id"] for c in cs]
    assert all(c["doc_id"] == "returns" for c in cs)


def check_no_empty_chunks(m):
    """a heading without text makes no chunk (ids stay consecutive)"""
    cs = m.chunk_markdown("# Title\n## A\n\n## B\nbody\n", "d")
    assert [c["text"] for c in cs] == ["body"], cs
    assert cs[0]["id"] == "d#0", cs[0]["id"]


def check_real_faq(m):
    """works on the real FAQ pages"""
    from learn.data import faq_docs
    total = sum(len(m.chunk_markdown(t, n)) for n, t in faq_docs().items())
    assert total >= 12, f"only {total} chunks from the FAQ pages"
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 1: split a markdown document into chunks that keep their heading path."""
from __future__ import annotations


def chunk_markdown(text: str, doc_id: str) -> list[dict]:
    """Return [{"id": "returns#0", "doc_id": "returns", "heading": "Returns > Return window", "text": "..."}, ...]"""
    chunks: list[dict] = []
    h1, h2, buf = "", "", []

    def flush():
        body = " ".join(" ".join(buf).split())
        if body:
            heading = " > ".join(x for x in (h1, h2) if x)
            chunks.append({"id": f"{doc_id}#{len(chunks)}", "doc_id": doc_id, "heading": heading, "text": body})
        buf.clear()

    for line in text.splitlines():
        if line.startswith("## "):
            flush()
            h2 = line[3:].strip()
        elif line.startswith("# "):
            flush()
            h1, h2 = line[2:].strip(), ""
        else:
            buf.append(line)
    flush()
    return chunks
```

</details>

## خروجی مورد انتظار
هر خط تیک ✓ می‌خورد و در آخر `PASSED step01_chunk_markdown`. قبل از نوشتن کد، خروجی همه‌ی خط‌ها `✗ not implemented yet: TODO ...` است؛ این طبیعی است.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `heading` برابر `Returns` است نه `Returns > Return window` | `h1` و `h2` را با ` > ` به هم نچسباندی |
| متن تکه شامل خط عنوان است | خط عنوان را هم به `buf` اضافه کرده‌ای |
| `id` از `#1` شروع می‌شود | شماره را از تعداد خط‌ها یا بخش‌ها گرفته‌ای، نه از `len(chunks)` |
| یک تکه‌ی خالی ساخته شده | قبل از ساختن تکه بررسی نکرده‌ای متن خالی نباشد |

## دام‌ها
- **فقط با یک سند آزمایش نکن.** سند واقعی گاهی عنوان ندارد یا خط خالی زیاد دارد.
- chunk بدون `heading` زمینه را از دست می‌دهد؛ بعداً heading را به متن اضافه می‌کنیم تا جست‌وجو بهتر شود (ایده‌ی contextual retrieval).

## چند تمرین اضافه (اختیاری)
1. عنوان سطح ۳ (`###`) را هم پشتیبانی کن.
2. اگر یک بخش بیش از ۱۲۰ کلمه بود، آن را به دو تکه بشکن.
3. روی `lessons/_data/faq/*.md` اجرا کن و تعداد تکه‌ها را چاپ کن.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p01_faq_search', 'step01_chunk_markdown')

---

# گام ۲: جست‌وجوی کلیدواژه‌ای با BM25

## هدف این گام
خودت BM25 را از صفر می‌نویسی (بدون کتابخانه). BM25 الگوریتم کلاسیک جست‌وجو است و هنوز در سیستم‌های واقعی (Elasticsearch، Lucene و RAG های production) به‌کار می‌رود، چون سریع، ارزان و قابل‌فهم است.

## پیش‌نیاز
گام ۱ (تکه‌ها). لازم نیست پاسخ گام ۱ درست باشد؛ این گام با لیست متن‌های ساده کار می‌کند.

## مفهوم: چطور یک سند را امتیاز بدهیم؟
سه ایده‌ی ساده پشت BM25 است:

1. **تکرار کلمه (term frequency, tf):** اگر کلمه‌ی «refund» در یک تکه چند بار آمده، احتمالاً درباره‌ی بازپرداخت است. ولی تکرار بیشتر ارزش کمتر دارد (۱۰ بار خیلی بهتر از ۹ بار نیست). BM25 این را با یک منحنی «اشباع» مدل می‌کند.
2. **نادر بودن کلمه (inverse document frequency, idf):** کلمه‌ی «the» در همه‌ی تکه‌هاست و هیچ چیز نمی‌گوید؛ «warranty» فقط در یکی است و خیلی گویاست. پس کلمه‌ی نادر وزن بیشتری دارد.
3. **طول سند:** تکه‌ی کوتاهی که کلمه را دارد، متمرکزتر از تکه‌ی بلندی است که همان کلمه را دارد. پس طول را نرمال می‌کنیم.

فرمول برای یک کلمه‌ی query و یک سند:

```
idf(t)  = ln(1 + (N - df + 0.5) / (df + 0.5))
score   = idf(t) * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len / avg_len))
```
- `N` تعداد تکه‌ها، `df` تعداد تکه‌هایی که کلمه را دارند، `tf` تکرار کلمه در این تکه
- `k1` (معمولاً 1.5) شدت اشباع تکرار، `b` (معمولاً 0.75) شدت نرمال‌سازی طول
- امتیاز کل سند = جمع امتیاز همه‌ی کلمه‌های query

## تمرین کد

1. `tokenize(text)`: متن را lowercase کن، فقط حرف و عدد را نگه دار (`re.findall(r"[a-z0-9]+", ...)`) و کلمه‌های بی‌ارزش (`the`, `a`, `is`, ...) را که در `STOP` آمده حذف کن.
2. `BM25.__init__`: توکن‌های هر سند، طول‌ها، میانگین طول و `df` هر کلمه را حساب کن.
3. `BM25.scores(query)`: امتیاز هر سند را به‌صورت لیست برگردان (طول لیست = تعداد سندها).
4. `BM25.search(query, k)`: اندیس k سند برتر را برگردان؛ سندهای با امتیاز صفر نباید در نتیجه باشند.

### کد، خط‌به‌خط
- `self.tf = [Counter(tokens) ...]` تکرار هر کلمه را در هر سند می‌دهد.
- `df` را با `Counter` و حلقه روی `set(tokens)` هر سند بساز تا هر سند برای هر کلمه فقط یک بار شمرده شود.
- در `scores` فقط برای کلمه‌هایی که در `df` هستند امتیاز بده؛ کلمه‌ی ناشناخته idf ندارد و نادیده گرفته می‌شود.
- برای ترتیب هم‌امتیازها از اندیس استفاده کن تا نتیجه قطعی (deterministic) باشد: `sorted(..., key=lambda i: (-s[i], i))`.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 2: BM25 from scratch."""
from __future__ import annotations

import math
import re
from collections import Counter

STOP = {"the", "a", "an", "is", "are", "do", "does", "i", "you", "my", "to", "of", "and", "or", "in", "on", "for", "can", "how", "what",
        "it", "we", "your", "with", "if", "be", "at", "by", "from", "this", "that"}


def tokenize(text: str) -> list[str]:
    # TODO 1: lowercase, keep only letters/digits (re.findall(r"[a-z0-9]+", ...)) and drop the words in STOP
    raise NotImplementedError("TODO 1: tokenize")


class BM25:
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        # TODO 2: self.tf = one Counter of tokens per doc; self.len = token count per doc; self.avg = mean length;
        #         self.df = Counter: in how many docs each token appears (count each doc once per token); self.n = number of docs
        raise NotImplementedError("TODO 2: index the documents")

    def idf(self, term: str) -> float:
        # TODO 3: ln(1 + (N - df + 0.5) / (df + 0.5))
        raise NotImplementedError("TODO 3: idf")

    def scores(self, query: str) -> list[float]:
        # TODO 4: for every doc add idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * len / avg)) for each query token
        #         that appears in the corpus (df > 0). Return one float per doc.
        raise NotImplementedError("TODO 4: score every document")

    def search(self, query: str, k: int = 3) -> list[int]:
        # TODO 5: indices of the k best documents, best first, ties by smaller index, score 0 excluded
        raise NotImplementedError("TODO 5: top-k")

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p01_faq_search', 'step02_bm25', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
DOCS = ["You can return items within 30 days of delivery",
        "We refund your money to the original payment method within 14 days",
        "Standard delivery takes 3 to 5 working days",
        "Furniture has a 2 year warranty against defects"]


def check_tokenize(m):
    """tokenize lowercases, keeps words and numbers, drops stop words"""
    assert m.tokenize("The Refund, is 14 days!") == ["refund", "14", "days"], m.tokenize("The Refund, is 14 days!")


def check_rare_words_weigh_more(m):
    """a rare word has a higher idf than a common one"""
    bm = m.BM25(DOCS)
    assert bm.idf("warranty") > bm.idf("within"), (bm.idf("warranty"), bm.idf("within"))
    assert bm.idf("within") > 0, "idf must stay positive"


def check_ranking(m):
    """the matching document comes first"""
    bm = m.BM25(DOCS)
    assert bm.search("refund money", k=2)[0] == 1, bm.search("refund money", k=2)
    assert bm.search("warranty defects")[0] == 3
    assert bm.search("standard working days")[0] == 2, bm.search("standard working days")


def check_zero_scores_excluded(m):
    """documents without any query word are not returned"""
    bm = m.BM25(DOCS)
    assert bm.search("bitcoin") == [], bm.search("bitcoin")
    assert len(bm.scores("bitcoin")) == 4 and sum(bm.scores("bitcoin")) == 0


def check_length_normalisation(m):
    """with the same tf, the shorter document scores higher"""
    bm = m.BM25(["refund", "refund policy of the shop is long and has many other words inside", "unrelated text"])
    s = bm.scores("refund")
    assert s[0] > s[1] > 0, s


def check_deterministic_ties(m):
    """equal scores are ordered by index"""
    bm = m.BM25(["alpha beta", "alpha beta", "gamma"])
    assert bm.search("alpha", k=2) == [0, 1]
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 2: BM25 from scratch."""
from __future__ import annotations

import math
import re
from collections import Counter

STOP = {"the", "a", "an", "is", "are", "do", "does", "i", "you", "my", "to", "of", "and", "or", "in", "on", "for", "can", "how", "what",
        "it", "we", "your", "with", "if", "be", "at", "by", "from", "this", "that"}


def tokenize(text: str) -> list[str]:
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]


class BM25:
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        toks = [tokenize(d) for d in docs]
        self.tf = [Counter(t) for t in toks]
        self.len = [len(t) for t in toks]
        self.n = len(docs)
        self.avg = sum(self.len) / self.n if self.n else 0.0
        self.df = Counter(w for t in toks for w in set(t))

    def idf(self, term: str) -> float:
        df = self.df.get(term, 0)
        return math.log(1 + (self.n - df + 0.5) / (df + 0.5))

    def scores(self, query: str) -> list[float]:
        out = [0.0] * self.n
        for term in tokenize(query):
            if self.df.get(term, 0) == 0:
                continue
            idf = self.idf(term)
            for i in range(self.n):
                tf = self.tf[i].get(term, 0)
                if tf:
                    out[i] += idf * tf * (self.k1 + 1) / (tf + self.k1 * (1 - self.b + self.b * self.len[i] / self.avg))
        return out

    def search(self, query: str, k: int = 3) -> list[int]:
        s = self.scores(query)
        return [i for i in sorted(range(self.n), key=lambda i: (-s[i], i)) if s[i] > 0][:k]
```

</details>

## خروجی مورد انتظار
پنج خط ✓ و `PASSED step02_bm25`.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| idf منفی یا صفر شد | فرمول را بدون `1 +` داخل لگاریتم نوشته‌ای؛ با کلمه‌های همه‌جا حاضر منفی می‌شود |
| سند بلندتر برنده شد | نرمال‌سازی طول (`b`) را نگذاشته‌ای |
| `ZeroDivisionError` | `avg_len` وقتی لیست سندها خالی است صفر می‌شود؛ حالت خالی را جدا مدیریت کن |
| یک query بی‌ربط سند برمی‌گرداند | صفر امتیازها را فیلتر نکرده‌ای |

## دام‌ها
- BM25 **معنا را نمی‌فهمد**: «sofa» و «couch» برایش دو کلمه‌ی بی‌ربط‌اند و غلط املایی کلمه را از بین می‌برد. این ضعف در گام ۴ با روش دوم جبران می‌شود.
- tokenize ساده برای فارسی/آلمانی کافی نیست (ریشه‌یابی و حروف خاص لازم است)؛ در پروژه‌ی p09 به آن می‌رسیم.

## تمرین اضافه
1. `k1` را 0.5 و 3.0 کن و ببین ترتیب نتایج یک query چطور عوض می‌شود.
2. `b=0` کن و ببین سندهای بلند چقدر بالا می‌آیند.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p01_faq_search', 'step02_bm25')

---

# گام ۳: اندازه‌گیری کیفیت جست‌وجو (recall@k و MRR)

## هدف این گام
قبل از «بهتر کردن» جست‌وجو باید بتوانی «بهتر» را عددی بسنجی. در این گام دو معیار استاندارد را می‌نویسی و یک تابع ارزیابی که هر جست‌وجویی را با مجموعه‌ی سؤال‌ها امتحان می‌کند.

## پیش‌نیاز
گام‌های ۱ و ۲ را بفهم؛ ولی این گام مستقل است و فقط با لیست شناسه‌ها کار می‌کند.

## مفهوم
**چرا ارزیابی؟** بدون عدد، «احساس می‌کنم بهتر شد» تنها دلیل توست و این معمولاً غلط است. هر تغییری (chunk جدید، الگوریتم جدید، مدل جدید) را با همان مجموعه‌ی سؤال‌ها می‌سنجیم.

هر سؤال یک **جواب درست** دارد: سندی که جواب در آن است (`relevant`). جست‌وجو لیستی مرتب از سندها برمی‌گرداند (`ranked`).

- **recall@k** : آیا حداقل یکی از سندهای درست در k نتیجه‌ی اول هست؟ (۱ یا ۰ برای هر سؤال؛ میانگین روی همه). اگر مدل زبانی k نتیجه‌ی اول را می‌بیند، recall@k می‌گوید «حداقل شانس جواب درست دادن داشته». اینجا اگر چند سند درست داریم، نسبت سندهای درست پیدا شده به کل آن‌ها را بگیر (`|ranked[:k] ∩ relevant| / |relevant|`).
- **MRR** (mean reciprocal rank): اگر اولین سند درست در رتبه‌ی r باشد، امتیاز `1/r`؛ اگر اصلاً نباشد ۰. میانگین روی سؤال‌ها. می‌گوید سند درست چقدر **بالا** آمده؛ رتبه‌ی ۱ یعنی ۱.۰، رتبه‌ی ۲ یعنی ۰.۵.

دو معیار مکمل‌اند: recall می‌گوید «پیدا شد؟» و MRR می‌گوید «کجا؟».

**سؤال‌های بی‌جواب:** سؤالی مثل «آیا چک می‌پذیرید؟» جواب درست در هیچ سندی ندارد (`relevant` خالی). برای جست‌وجو معیار recall ندارد؛ آن‌ها را در `evaluate` کنار بگذار (در گام‌های بعد برای امتناع، abstain، لازم‌شان داریم).

## تمرین کد

1. `recall_at_k(ranked, relevant, k)` : نسبتی از `relevant` که در k نتیجه‌ی اول است. اگر `relevant` خالی بود `0.0`.
2. `reciprocal_rank(ranked, relevant)` : `1/رتبه‌ی اولین درست` (رتبه از ۱ شروع می‌شود)، وگرنه `0.0`.
3. `evaluate(search, questions, k=3)`:
   - `search(question)` لیست شناسه‌ی سندها (مرتب) برمی‌گرداند؛ ممکن است یک سند چند بار بیاید (چند تکه از یک سند) و باید یکتا شود به ترتیب اولین ظهور.
   - `questions` لیست دیکشنری با کلیدهای `question` و `relevant` (رشته‌ی خالی = بی‌جواب).
   - خروجی: `{"recall@k": ..., "mrr": ..., "misses": [سؤال‌هایی که recall صفر داشتند], "n": تعداد سؤال‌های قابل ارزیابی}`

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 3: retrieval metrics."""
from __future__ import annotations


def recall_at_k(ranked: list[str], relevant: set[str], k: int) -> float:
    # TODO 1: fraction of `relevant` that appears in ranked[:k]; 0.0 when relevant is empty
    raise NotImplementedError("TODO 1: recall@k")


def reciprocal_rank(ranked: list[str], relevant: set[str]) -> float:
    # TODO 2: 1 / (rank of the first relevant item), rank starting at 1; 0.0 if none
    raise NotImplementedError("TODO 2: reciprocal rank")


def evaluate(search, questions: list[dict], k: int = 3) -> dict:
    """search(question) -> ranked list of doc ids (may contain duplicates).
    questions: [{"question": str, "relevant": "returns" or ""}] ("" = unanswerable, skip it)."""
    # TODO 3: for every answerable question: dedupe the ranking keeping first occurrences, compute both metrics,
    #         collect the question text in `misses` when recall@k == 0. Return the means.
    raise NotImplementedError("TODO 3: evaluate")

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p01_faq_search', 'step03_metrics', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
def check_recall_single(m):
    """recall@k with one relevant document"""
    assert m.recall_at_k(["a", "b", "c"], {"b"}, 2) == 1.0
    assert m.recall_at_k(["a", "b", "c"], {"c"}, 2) == 0.0


def check_recall_multiple(m):
    """recall@k is a fraction when several are relevant"""
    assert m.recall_at_k(["a", "b", "c"], {"a", "c"}, 2) == 0.5
    assert m.recall_at_k(["a"], set(), 3) == 0.0


def check_reciprocal_rank(m):
    """rank 1 -> 1.0, rank 2 -> 0.5, missing -> 0"""
    assert m.reciprocal_rank(["x", "y"], {"x"}) == 1.0
    assert m.reciprocal_rank(["x", "y"], {"y"}) == 0.5
    assert m.reciprocal_rank(["x", "y"], {"z"}) == 0.0


def check_evaluate_means(m):
    """evaluate averages over answerable questions only"""
    qs = [{"question": "q1", "relevant": "a"}, {"question": "q2", "relevant": "b"}, {"question": "q3", "relevant": ""}]
    ranks = {"q1": ["a", "b"], "q2": ["x", "y", "b"], "q3": ["a"]}
    out = m.evaluate(lambda q: ranks[q], qs, k=2)
    assert out["n"] == 2, out
    assert out["recall@2"] == 0.5, out
    assert abs(out["mrr"] - (1 + 1 / 3) / 2) < 1e-9, out
    assert out["misses"] == ["q2"], out


def check_duplicates_collapse(m):
    """several chunks of one document count once, in first-seen order"""
    qs = [{"question": "q", "relevant": "b"}]
    out = m.evaluate(lambda q: ["a", "a", "a", "b"], qs, k=2)
    assert out["recall@2"] == 1.0, f"duplicates pushed the answer out of the top 2: {out}"
    assert out["mrr"] == 0.5, out


def check_empty_is_safe(m):
    """no answerable question -> zeros, no crash"""
    out = m.evaluate(lambda q: [], [{"question": "q", "relevant": ""}], k=3)
    assert out["n"] == 0 and out["mrr"] == 0.0
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 3: retrieval metrics."""
from __future__ import annotations


def recall_at_k(ranked: list[str], relevant: set[str], k: int) -> float:
    if not relevant:
        return 0.0
    return len(set(ranked[:k]) & relevant) / len(relevant)


def reciprocal_rank(ranked: list[str], relevant: set[str]) -> float:
    for i, doc in enumerate(ranked):
        if doc in relevant:
            return 1 / (i + 1)
    return 0.0


def evaluate(search, questions: list[dict], k: int = 3) -> dict:
    rec, rr, misses = [], [], []
    for q in questions:
        if not q["relevant"]:
            continue
        ranked = list(dict.fromkeys(search(q["question"])))
        rel = {q["relevant"]}
        r = recall_at_k(ranked, rel, k)
        rec.append(r)
        rr.append(reciprocal_rank(ranked, rel))
        if r == 0:
            misses.append(q["question"])
    n = len(rec)
    return {f"recall@{k}": sum(rec) / n if n else 0.0, "mrr": sum(rr) / n if n else 0.0, "misses": misses, "n": n}
```

</details>

## خروجی مورد انتظار
شش خط ✓.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| MRR برای رتبه‌ی ۱ برابر ۰.۵ شد | رتبه را از ۰ شمرده‌ای؛ باید `index + 1` باشد |
| `evaluate` سؤال بی‌جواب را هم شمرد | `relevant == ""` را فیلتر نکرده‌ای |
| recall بیش از ۱ شد | سند تکراری را یکتا نکرده‌ای |

## دام‌ها
- **مجموعه‌ی ارزیابی کوچک** (اینجا ۱۲ سؤال) فقط برای یادگیری است. در پروژه‌ی واقعی حداقل ۵۰ تا ۱۰۰ سؤال بساز و اختلاف کوچک را جدی نگیر.
- سؤال‌های ارزیابی را از روی **سؤال‌های واقعی کاربر** بساز، نه از جمله‌های خود سند؛ وگرنه BM25 همیشه برنده است.
- **نشت (leakage):** اگر prompt یا قاعده‌هایت را روی همین سؤال‌ها تنظیم کنی، عدد بیش از حد خوش‌بینانه می‌شود. بعداً بخشی از سؤال‌ها را کنار بگذار.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p01_faq_search', 'step03_metrics')

---

# گام ۴: دو جست‌وجو بهتر از یکی است (hybrid با RRF)

## هدف این گام
یک جست‌وجوی دوم می‌نویسی که به غلط املایی مقاوم است (شباهت حروف)، و دو نتیجه را با **RRF** (Reciprocal Rank Fusion) ترکیب می‌کنی. نتیجه را روی سؤال‌های واقعی می‌سنجی.

## پیش‌نیاز

## مفهوم
هر جست‌وجو یک نقطه‌ضعف دارد:
- **BM25** دقیق است ولی کلمه‌ی غلط (`standrd`) یا هم‌معنی (`couch`/`sofa`) را نمی‌فهمد.
- **شباهت حروف (character trigram)** کلمه‌ها را به سه‌حرفی‌های هم‌پوشان می‌شکند (`refund` → `ref`, `efu`, `fun`, `und`) و شباهت را با Jaccard می‌سنجد. غلط املایی کوچک فقط چند سه‌حرفی را خراب می‌کند. ولی معنا را نمی‌فهمد و برای کلمه‌های رایج هم امتیاز می‌دهد.
- **embedding** (در پروژه‌ی اصلی، بعداً) معنا را می‌فهمد ولی گران‌تر و گاهی در کلمه‌های دقیق (کد محصول، عدد) ضعیف است.

ترکیب نتیجه‌ها ساده نیست چون امتیازها هم‌مقیاس نیستند (BM25 عددی مثل 7.2 می‌دهد و Jaccard عددی بین 0 و 1). **RRF** امتیاز را کنار می‌گذارد و فقط **رتبه** را به‌کار می‌برد:

```
rrf_score(doc) = Σ over rankers   1 / (k + rank_in_that_ranker)      (rank از 1، معمولاً k = 60)
```
سندی که در هر دو فهرست بالاست بیشترین امتیاز را می‌گیرد. چون به مقیاس امتیاز حساس نیست، بدون تنظیم کار می‌کند و در عمل روش ترکیب پیش‌فرض بسیاری از سیستم‌هاست.

## تمرین کد

1. `trigrams(text)` : مجموعه‌ی سه‌حرفی‌های هر کلمه‌ی متن (کلمه‌ها را با فاصله‌ی ابتدا و انتها، مثل `" refund "` جدا کن تا ابتدا و انتهای کلمه هم مهم باشد).
2. `trigram_rank(query, texts)` : اندیس متن‌ها را به‌ترتیب شباهت Jaccard (`|A∩B| / |A∪B|`) نزولی برگردان؛ فقط شباهت‌های بزرگ‌تر از صفر؛ ترتیب هم‌امتیازها با اندیس.
3. `rrf(rankings, k=60)` : `rankings` لیستی از فهرست‌های اندیس (هر کدام مرتب) است؛ فهرست ترکیبی را برگردان (امتیاز نزولی، هم‌امتیاز: اندیس کوچک‌تر).
4. `hybrid(query, bm25, texts, top=3)` : از BM25 (روی همان `texts`) و `trigram_rank` رتبه بگیر، با `rrf` ترکیب کن، `top` اول را برگردان.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 4: character-trigram search and Reciprocal Rank Fusion."""
from __future__ import annotations

import math
import re
from collections import Counter


def trigrams(text: str) -> set[str]:
    # TODO 1: for every word of the lowercase text, pad it as f" {word} " and collect all 3-character windows
    raise NotImplementedError("TODO 1: trigrams")


def trigram_rank(query: str, texts: list[str]) -> list[int]:
    # TODO 2: Jaccard similarity |A & B| / |A | B| between trigrams(query) and each text; indices by similarity
    #         descending, ties by smaller index, similarity 0 excluded
    raise NotImplementedError("TODO 2: trigram ranking")


def rrf(rankings: list[list[int]], k: int = 60) -> list[int]:
    # TODO 3: score(doc) = sum over rankings of 1 / (k + rank), rank starting at 1; only docs that appear somewhere;
    #         best first, ties by smaller index
    raise NotImplementedError("TODO 3: reciprocal rank fusion")


def hybrid(query: str, bm25, texts: list[str], top: int = 3) -> list[int]:
    # TODO 4: ranking from bm25.search(query, k=len(texts)) and from trigram_rank, fused with rrf, first `top`
    raise NotImplementedError("TODO 4: hybrid search")


# ---- helper so the demo below runs: a compact BM25 (you wrote the real one in step 2) ----
class SimpleBM25:
    def __init__(self, docs):
        self.toks = [re.findall(r"[a-z0-9]+", d.lower()) for d in docs]
        self.df = Counter(w for t in self.toks for w in set(t))
        self.n, self.avg = len(docs), sum(map(len, self.toks)) / max(len(docs), 1)

    def search(self, query, k=3):
        q = re.findall(r"[a-z0-9]+", query.lower())
        sc = []
        for t in self.toks:
            tf = Counter(t)
            s = 0.0
            for w in q:
                if tf[w]:
                    idf = math.log(1 + (self.n - self.df[w] + .5) / (self.df[w] + .5))
                    s += idf * tf[w] * 2.5 / (tf[w] + 1.5 * (.25 + .75 * len(t) / self.avg))
            sc.append(s)
        return [i for i in sorted(range(self.n), key=lambda i: (-sc[i], i)) if sc[i] > 0][:k]

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
import sys
sys.path.insert(0, ".")
from learn.data import faq_docs, faq_questions
texts, owner = [], []
for name, doc in faq_docs().items():
    for part in doc.split("\n## ")[1:]:
        texts.append(part), owner.append(name)
bm = SimpleBM25(texts)
qs = faq_questions() + [{"question": "how fast is standrd delivry", "relevant": "shipping", "answerable": "yes"}]
for label, fn in [("bm25", lambda q: bm.search(q, k=len(texts))), ("trigram", lambda q: trigram_rank(q, texts)),
                  ("hybrid", lambda q: hybrid(q, bm, texts, top=len(texts)))]:
    hit = [q for q in qs if q["relevant"] and q["relevant"] in {owner[i] for i in fn(q["question"])[:3]}]
    print(f"{label:8s} recall@3 = {len(hit) / sum(1 for q in qs if q['relevant']):.2f}")

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p01_faq_search', 'step04_hybrid_rrf', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
def check_trigrams(m):
    """trigrams of a word are padded with spaces"""
    assert m.trigrams("ab") == {" ab", "ab "}, m.trigrams("ab")
    assert "ref" in m.trigrams("Refund") and " re" in m.trigrams("Refund") and "nd " in m.trigrams("Refund")


def check_trigram_typo(m):
    """a misspelled word still finds its document"""
    texts = ["standard delivery takes three days", "we refund the money", "warranty for furniture"]
    assert m.trigram_rank("standrd delivry", texts)[0] == 0, m.trigram_rank("standrd delivry", texts)


def check_trigram_zero_excluded(m):
    """no shared trigram -> not returned"""
    assert m.trigram_rank("qqq", ["abc def"]) == []


def check_rrf_order(m):
    """a document high in both lists wins"""
    assert m.rrf([[2, 0, 1], [2, 1, 0]])[0] == 2
    assert m.rrf([[0, 1], [1, 0]]) == [0, 1], "tie -> smaller index"


def check_rrf_union_and_scores(m):
    """documents in only one list are kept; scores use 1/(k+rank)"""
    out = m.rrf([[5], [7, 5]], k=60)
    assert out == [5, 7], out
    assert m.rrf([[1, 2]], k=1) == [1, 2]


def check_hybrid_recovers_typos(m):
    """hybrid finds the right text where BM25 alone fails"""
    texts = ["Standard delivery takes 3 to 5 working days.", "We send your money back within 14 days.",
             "Furniture has a 2 year warranty.", "Returns are free within 30 days."]
    bm = m.SimpleBM25(texts)
    q = "how fast is standrd delivry"
    assert bm.search(q, k=4)[:1] != [0], "(demo precondition) BM25 alone should miss the typo query"
    assert m.hybrid(q, bm, texts, top=1) == [0], m.hybrid(q, bm, texts, top=1)
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 4: character-trigram search and Reciprocal Rank Fusion."""
from __future__ import annotations

import math
import re
from collections import Counter


def trigrams(text: str) -> set[str]:
    out: set[str] = set()
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        p = f" {w} "
        out |= {p[i:i + 3] for i in range(len(p) - 2)}
    return out


def trigram_rank(query: str, texts: list[str]) -> list[int]:
    q = trigrams(query)
    sims = []
    for t in texts:
        g = trigrams(t)
        sims.append(len(q & g) / len(q | g) if q | g else 0.0)
    return [i for i in sorted(range(len(texts)), key=lambda i: (-sims[i], i)) if sims[i] > 0]


def rrf(rankings: list[list[int]], k: int = 60) -> list[int]:
    score: dict[int, float] = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking, start=1):
            score[doc] = score.get(doc, 0.0) + 1 / (k + rank)
    return sorted(score, key=lambda d: (-score[d], d))


def hybrid(query: str, bm25, texts: list[str], top: int = 3) -> list[int]:
    return rrf([bm25.search(query, k=len(texts)), trigram_rank(query, texts)])[:top]


# ---- helper so the demo below runs: a compact BM25 (you wrote the real one in step 2) ----
class SimpleBM25:
    def __init__(self, docs):
        self.toks = [re.findall(r"[a-z0-9]+", d.lower()) for d in docs]
        self.df = Counter(w for t in self.toks for w in set(t))
        self.n, self.avg = len(docs), sum(map(len, self.toks)) / max(len(docs), 1)

    def search(self, query, k=3):
        q = re.findall(r"[a-z0-9]+", query.lower())
        sc = []
        for t in self.toks:
            tf = Counter(t)
            s = 0.0
            for w in q:
                if tf[w]:
                    idf = math.log(1 + (self.n - self.df[w] + .5) / (self.df[w] + .5))
                    s += idf * tf[w] * 2.5 / (tf[w] + 1.5 * (.25 + .75 * len(t) / self.avg))
            sc.append(s)
        return [i for i in sorted(range(self.n), key=lambda i: (-sc[i], i)) if sc[i] > 0][:k]
```

</details>

## خروجی مورد انتظار
شش ✓. اجرای فایل جدول کوچکی از recall@3 برای BM25 و trigram و hybrid چاپ می‌کند؛ معمولاً hybrid نزدیک به بهترین روش تکی است و روی سؤال غلط املایی از BM25 تنها بهتر؛ ولی تضمینی نیست (دام پایین).

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| `rrf` سند بدون رتبه را هم می‌آورد | فقط سندهایی که در حداقل یک فهرست هستند باید باشند |
| نتیجه هر بار فرق می‌کند | ترتیب هم‌امتیازها را با اندیس قطعی نکرده‌ای |
| trigram همه‌ی سندها را برمی‌گرداند | شباهت صفر را حذف نکرده‌ای |

## دام‌ها
- RRF رتبه‌ها را جمع می‌کند و کیفیت روش‌ها را نمی‌سنجد. اگر یک جست‌وجو بد است، ترکیب می‌تواند خرابش کند؛ همیشه با گام ۳ بسنج.
- `k=60` عدد قراردادی است؛ k کوچک‌تر رتبه‌های اول را مهم‌تر می‌کند.

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p01_faq_search', 'step04_hybrid_rrf')

---

# گام ۵: جواب با ارجاع به منبع و حق «نمی‌دانم»

## هدف این گام
آخرین حلقه‌ی یک سیستم FAQ: تکه‌های پیدا‌شده را به مدل زبانی می‌دهی، جواب با **ارجاع** می‌گیری، ارجاع‌های ساختگی را حذف می‌کنی و وقتی شواهد نیست **امتناع** می‌کنی. بعد همان تابع را با مدل واقعی اجرا می‌کنی: Ollama یا Claude یا مدل‌های OpenCode.

## پیش‌نیاز
گام‌های قبل مفهومی‌اند. این گام فقط یک شیء `llm` با متد `chat(messages)` می‌خواهد؛ در تست یک مدل ساختگی است و در اجرای واقعی `learn.llm.get()`.

## مفهوم
مدل زبانی خوب می‌نویسد ولی دو خطای خطرناک دارد: **ساختن واقعیت** (hallucination) و **جواب دادن وقتی نباید**. دفاع‌ها که در **کد** هستند، نه در prompt:

1. **جواب فقط از شواهد:** prompt می‌گوید فقط از متن‌های داده‌شده استفاده کن و بعد از هر ادعا شناسه‌ی منبع را در کروشه بنویس `[returns#0]`.
2. **ارجاع باید واقعی باشد:** بعد از جواب، شناسه‌های داخل کروشه را با شناسه‌های تکه‌هایی که فرستادیم مقایسه می‌کنیم. شناسه‌ی ساختگی حذف می‌شود.
3. **بی‌شاهد = بی‌جواب:** اگر هیچ تکه‌ای پیدا نشد، مدل را صدا نمی‌زنیم (هم ارزان‌تر، هم امن‌تر). اگر مدل جواب داد ولی هیچ ارجاع معتبری نداشت، جواب را نمی‌پذیریم.

نتیجه یک دیکشنری است: `{"answer": متن یا None, "citations": [...], "abstained": bool}`.

## تمرین کد

1. `build_messages(question, chunks)` : لیست پیام‌ها (`system` و `user`). `chunks` لیست دیکشنری‌های گام ۱ است (`id`, `heading`, `text`). متن user باید هر تکه را به شکل `[id] heading: text` و بعد سؤال را داشته باشد.
2. `extract_citations(text, allowed_ids)` : همه‌ی `[...]`ها را پیدا کن، فقط آن‌هایی که در `allowed_ids` هستند، بدون تکرار و به ترتیب ظهور.
3. `answer(question, chunks, llm)` : مراحل بالا. جواب مدل از `llm.chat(messages).content` می‌آید.

### ✍️ کد تو
کد زیر را ببین، TODO ها را کامل کن و سلول را اجرا کن (Shift+Enter). این‌جا می‌نویسی، نه در فایل دیگر.

In [ ]:
"""Step 5: grounded answer with citations and abstention."""
from __future__ import annotations

import re

SYSTEM = ("You answer customer questions using ONLY the sources provided. After every claim write the id of its source "
          "in square brackets, like [returns#0]. If the sources do not contain the answer, say you do not know.")


def build_messages(question: str, chunks: list[dict]) -> list[dict]:
    # TODO 1: [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ...}] where the user text lists every
    #         chunk as "[id] heading: text" (one per line) followed by "Question: ..."
    raise NotImplementedError("TODO 1: build the prompt")


def extract_citations(text: str, allowed_ids: set[str]) -> list[str]:
    # TODO 2: every [...] in the text, keep only ids in allowed_ids, no duplicates, in order of appearance
    raise NotImplementedError("TODO 2: extract valid citations")


def answer(question: str, chunks: list[dict], llm) -> dict:
    # TODO 3: no chunks -> abstain WITHOUT calling the model; otherwise call llm.chat(messages).content, extract the
    #         valid citations; no valid citation -> abstain. Return {"answer": str | None, "citations": [...], "abstained": bool}
    raise NotImplementedError("TODO 3: answer or abstain")

### ▶ اجرای نمایشی
اول کد را بخوان، بعد اجرا کن. این سلول از توابعی که بالا نوشتی استفاده می‌کند.

In [ ]:
from learn import llm as learn_llm
from learn.data import faq_chunks, faq_questions, retrieve

chunks, model = faq_chunks(), learn_llm.get()          # offline by default; see LEARN_PROVIDER in lessons/00_setup
for q in faq_questions():
    out = answer(q["question"], retrieve(q["question"], chunks), model)
    print(f"{q['question'][:45]:45s} -> {'ABSTAINED' if out['abstained'] else out['answer'][:70]!r:70s} {out['citations']}")

### ✅ بررسی خودکار
تست‌ها را روی چیزی که در سلول‌های بالا نوشتی اجرا می‌کند. ✗ = هنوز نه؛ پیام را بخوان و سلول کد را اصلاح و دوباره اجرا کن.

In [ ]:
from learn.notebook import check_cell
check_cell('p01_faq_search', 'step05_answer_citations', globals())

<details><summary><b>🔍 این تست‌ها دقیقاً چه چیزی را می‌سنجند؟</b> (کد تست)</summary>

```python
CHUNKS = [{"id": "returns#0", "heading": "Returns > Return window", "text": "You can return most items within 30 days."},
          {"id": "refunds#0", "heading": "Refunds > Timing", "text": "Money comes back within 14 days."}]


class Fake:
    """Stands in for a model: returns a prepared text and counts calls."""

    def __init__(self, text):
        self.text, self.calls, self.messages = text, 0, None

    def chat(self, messages, tools=None, json_schema=None):
        self.calls += 1
        self.messages = messages

        class R:
            content = self.text
        return R()


def check_prompt_lists_sources(m):
    """the prompt contains every source with its id, and the question"""
    msgs = m.build_messages("how long?", CHUNKS)
    assert [x["role"] for x in msgs] == ["system", "user"], [x["role"] for x in msgs]
    u = msgs[1]["content"]
    assert "[returns#0]" in u and "30 days" in u and "[refunds#0]" in u and "how long?" in u, u


def check_extract_valid_only(m):
    """only real ids are kept, once each, in order"""
    got = m.extract_citations("A [refunds#0] and B [fake#9] and again [refunds#0] and [returns#0]", {"returns#0", "refunds#0"})
    assert got == ["refunds#0", "returns#0"], got


def check_answer_with_citation(m):
    """a cited answer is accepted"""
    llm = Fake("You have 30 days [returns#0].")
    out = m.answer("how long?", CHUNKS, llm)
    assert out == {"answer": "You have 30 days [returns#0].", "citations": ["returns#0"], "abstained": False}, out


def check_fabricated_citation_rejected(m):
    """an answer that cites only invented ids is rejected"""
    out = m.answer("q", CHUNKS, Fake("Sure, 90 days [policy#42]."))
    assert out["abstained"] and out["answer"] is None, out


def check_no_chunks_no_model_call(m):
    """without sources the model is not even called"""
    llm = Fake("anything [x#1]")
    out = m.answer("q", [], llm)
    assert out["abstained"] is True and llm.calls == 0, (out, llm.calls)


def check_uncited_answer_rejected(m):
    """an answer without any citation is not trusted"""
    out = m.answer("q", CHUNKS, Fake("I think it is 30 days."))
    assert out["abstained"] and out["citations"] == [], out
```

</details>

<details><summary><b>💡 جواب مرجع</b> (بعد از یک تلاش واقعی باز کن)</summary>

```python
"""Step 5: grounded answer with citations and abstention."""
from __future__ import annotations

import re

SYSTEM = ("You answer customer questions using ONLY the sources provided. After every claim write the id of its source "
          "in square brackets, like [returns#0]. If the sources do not contain the answer, say you do not know.")


def build_messages(question: str, chunks: list[dict]) -> list[dict]:
    sources = "\n".join(f"[{c['id']}] {c['heading']}: {c['text']}" for c in chunks)
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"Sources:\n{sources}\n\nQuestion: {question}"}]


def extract_citations(text: str, allowed_ids: set[str]) -> list[str]:
    seen: list[str] = []
    for cid in re.findall(r"\[([^\]]+)\]", text):
        if cid in allowed_ids and cid not in seen:
            seen.append(cid)
    return seen


def answer(question: str, chunks: list[dict], llm) -> dict:
    if not chunks:
        return {"answer": None, "citations": [], "abstained": True}
    text = llm.chat(build_messages(question, chunks)).content
    cites = extract_citations(text, {c["id"] for c in chunks})
    if not cites:
        return {"answer": None, "citations": [], "abstained": True}
    return {"answer": text, "citations": cites, "abstained": False}
```

</details>

## خروجی مورد انتظار
تست‌ها شش ✓. با مدل واقعی برای «how long do I have to send back a sofa» باید جوابی مثل «30 days [returns#0]» ببینی؛ با سؤال «can I pay by cheque» باید **امتناع** (`ABSTAINED`) یا جواب بدون ارجاع معتبر رد شود.

## اگر فرق داشت
| علامت | علت رایج |
|---|---|
| ارجاع ساختگی در خروجی ماند | `extract_citations` را با `allowed_ids` فیلتر نکرده‌ای |
| با chunk خالی هم مدل صدا زده شد | شرط `if not chunks` را قبل از `llm.chat` نگذاشته‌ای |
| `opencode failed` | OpenCode نصب نیست یا به یک provider وارد نشده‌ای |
| `ANTHROPIC_API_KEY is not set` | کلید در `.env` نیست (اشتراک Claude.ai کلید API نیست) |

## دام‌ها
- «به مدل بگو اگر نمی‌دانی نگو» کافی نیست؛ بدون بررسی کد، مدل گاهی باز هم جواب می‌دهد. بررسی ارجاع یک دفاع قابل‌تست است.
- ارجاع معتبر ≠ جواب درست. مدل می‌تواند به تکه‌ی درست اشاره کند و چیز دیگری بگوید. راستی‌آزمایی (verify) را در درس p22 می‌بینی.
- داده‌ی واقعی مشتری را به مدل ابری نفرست مگر اجازه داشته باشی (بخش حریم داده در 00_setup).

## جمع‌بندی پروژه‌ی p01
حالا یک FAQ search کامل داری: chunk → BM25 + trigram → RRF → جواب با ارجاع. قدم بعد: [درس p22 Agentic RAG](#) که یاد می‌گیرد جست‌وجو را دوباره امتحان کند و بداند کی «نمی‌دانم» بگوید، و بعد جایگزینی با `labkit.retrieval.Index` (BM25 + embedding + reranker واقعی).

### 📝 امتحان کوتاه
به هر سؤال با شماره‌ی گزینه جواب بده.

In [ ]:
from learn.notebook import quiz_cell
quiz_cell('p01_faq_search', 'step05_answer_citations')

---
## 🏁 گزارش نهایی درس
نتیجه‌ی تست و امتحان هر گام، و آنچه هنوز مانده.

In [ ]:
from learn.notebook import lesson_report
lesson_report('p01_faq_search')